# We will use Autograd code to build an MLP from scratch

In [3]:
import math
import random

class Value:

  def __init__(self, val:float, name:str = "", children:set = set()):
    self.val = val
    self.grad = 0
    self._backprop = lambda: None
    self.name = name
    self.children = children
  def __repr__(self):
    return str(self.val)

  def __add__(self, other):
    if not isinstance(other, Value):
      other = Value(other)
    res =  Value(self.val + other.val, f"({self.name}+{other.name})", set([self, other]))

    def _backprop():
      self.grad += res.grad
      other.grad += res.grad

    res._backprop = _backprop
    return res
  
  def __radd__(self, other):
    return self + other
  
  def __rmul__(self, other):
    return self * other
  
  def __neg__(self):
    return -1 * self
  
  def __sub__(self, other):
    return self + (-other)
  
  def __mul__(self, other):
    if not isinstance(other, Value):
      other = Value(other)
    res = Value(self.val * other.val, f"({self.name}*{other.name})", set([self, other]))

    def _backprop():
      self.grad += res.grad * other.val
      other.grad += res.grad * self.val

    res._backprop = _backprop
    return res
  
  def __pow__(self, other):
    if isinstance(other, float) or isinstance(other, int):
      other = Value(other)

    res = Value(self.val ** other.val, f"({self.name}^{other.name})", set([self, other]))

    def _backprop():
      self.grad += other.val * self.val ** (other.val - 1) * res.grad

    res._backprop = _backprop
    return res
  
  def __truediv__(self, other):
    return self * other ** (-1)
  
  def exp(self):
    res = Value(math.e ** self.val, f"(e^{self.name})", set([self]))

    def _backprop():
      self.grad += math.e ** self.val * res.grad
    res._backprop = _backprop
    return res
  
  def tanh(self):
    x = self.val
    res = Value(((math.e**x - math.e**(-x)) / (math.e**x + math.e**(-x))), f"tanh({self.name})", set([self]))

    def _backprop():
      self.grad += (1 - (res.val)**2) * res.grad

    res._backprop = _backprop
    return res

def backprop(root):
  root.grad = 1

  topo_lst = []
  visited = set()

  def topo(cur):
    if cur in visited:
      return
    
    visited.add(cur)
    for node in cur.children:
      topo(node)
    topo_lst.append(cur)

  topo(root)

  for node in reversed(topo_lst):
    node._backprop()

## First define a single neuron

Take in inputs, perform the linear operation with weights and sum bias. Then apply non-linearity.

In [4]:
class Neuron:
    def __init__(self, in_dim, nonlinearity=True):
        self.w = [Value(random.uniform(-1, 1), name=f"w{i}") for i in range(in_dim)]
        self.b = Value(0.0, name="b")
        self.nonlinearity = nonlinearity

    def __call__(self, x):
        # perform w * x + b when using neuron
        # performing dot products with pure Python
        out = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return out.tanh() if self.nonlinearity else out

    def parameters(self):
        return self.w + [self.b]

In [6]:
x = [2,3]
n = Neuron(2)
n(x)

0.9743818345095936

## A single layer is just a list of neurons

In [7]:
class Layer:
    def __init__(self, in_dim, out_dim, nonlinearity=True):
        self.neurons = [Neuron(in_dim, nonlinearity=nonlinearity) for _ in range(out_dim)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

In [9]:
l = Layer(2,3)
l(x)

[0.9017481329641762, 0.9961192933232134, 0.9401078807540135]

## Layers feed into each other sequentially to construct an MLP

In [18]:
class MLP:
    def __init__(self, in_dim, layer_sizes):
        sizes = [in_dim] + layer_sizes
        self.layers = []
        for i in range(len(layer_sizes)):
            # last layer usually linear
            nonlinearity = (i != len(layer_sizes) - 1)
            self.layers.append(Layer(sizes[i], sizes[i + 1], nonlinearity=nonlinearity))

    def __call__(self, x):
        if not all(isinstance(xi, Value) for xi in x):
            x = [Value(xi) for xi in x]

        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

    def zero_grad(self):
        """
            Zero out weights before backward pass!
            If we don't do this, may have too fast a descent and overshoot!
            - Also because we want only the grad of the current batch to matter.
        """
        for p in self.parameters():
            p.grad = 0.0

In [10]:
x_1 = [2,3,-1]
m = MLP(3, [4,4,1]) # 3 input neurons, 2 hidden layers of 4, one output
m(x_1)

[1.3966497217155152]

In [12]:
def mse_loss(target, pred):
    loss = Value(0.0)
    for t, p in zip(target, pred):
        if not isinstance(t, Value):
            t = Value(t)
        if not isinstance(p, Value):
            p = Value(p)
        loss = loss + (p - t) ** 2
    return loss / len(target)

In [ ]:
mlp = MLP(in_dim=2, layer_sizes=[8, 4])
lr = 0.01

X = [
    [1.0, 2.0],
    [2.0, 1.0],
    [-1.0, 1.0],
    [0.5, -2.0],
]

Y = [
    [-2.0, 2.0, -4.0, 4.0],
    [-1.0, 1.0, -2.0, 2.0],
    [0.0, -1.0, 1.0, -1.0],
    [3.0, -3.0, 1.0, -1.0],
]

for epoch in range(200):
    total_loss = Value(0.0)

    for x, y in zip(X, Y):
        # forward pass
        pred = mlp(x)
        total_loss = total_loss + mse_loss(y, pred)

    total_loss = total_loss / len(X)

    mlp.zero_grad()
    # backward pass
    backprop(total_loss)    
    for p in mlp.parameters():
        p.val -= lr * p.grad

    if epoch % 20 == 0:
        print(f"epoch {epoch}, loss = {total_loss.val:.4f}")

epoch 0, loss = 4.4859
epoch 20, loss = 2.7194
epoch 40, loss = 1.9202
epoch 60, loss = 1.4259
epoch 80, loss = 1.0832
epoch 100, loss = 0.8372
epoch 120, loss = 0.6616
epoch 140, loss = 0.5384
epoch 160, loss = 0.4528
epoch 180, loss = 0.3927


In [16]:
# our own mini network!
len(mlp.parameters())

60

## Loss goes down, so the network is successfully learning!

p.s. a lot of the mistakes here were caught by Andrej Karpathy in the following Tweet: https://x.com/karpathy/status/1013244313327681536?s=20